# 01 — Physics Concept Extraction & Validation  ·  Critical Fix (M35 → concepts) + Gate G2 + CC2

**Current project direction:** physics-grounded, *label-free* acoustic **concept bottleneck** + faithfulness audit.
This notebook is the **first Critical Fix**: it turns the M35 physics DSP into standalone, clinically-named
**per-cycle concept extractors**, runs them on ICBHI (official 60/40 split), and **validates them against ICBHI's
crackle/wheeze cycle labels (Gate G2)**. It also runs the cheap **device-structure check (CC2)**.

**What "pass" looks like (G2):** the continuous `crackle_presence` / `wheeze_presence` concepts should track the
ICBHI labels with AUROC clearly above 0.5. If they do, the physics concepts are clinically meaningful and you may
build the bottleneck (notebook 02). If not, fall back to the minimal reliable concept set.

Outputs: `concepts_all.npz` (concept vectors + metadata), `concept_validation_report.json`, device report.


In [1]:
# --- OFFICIAL ICBHI split -----------------------------------------------------
# Do NOT hand-type patient IDs here. The published split assigns RECORDINGS, not
# patients, and a hand-typed 47-patient list was 15/47 correct (2026-08-26 run): it put
# 32 train patients into test, left 34 test patients in train, and inverted the disease
# class balance between the two sides. Load the committed file that ships inside the
# owmtl package instead. Audit + rationale: Asif's/audit/official_split.py
import sys, os, collections
OWMTL_PKG = "/kaggle/input/datasets/barshonbasak/owmtl-package"
sys.path.insert(0, OWMTL_PKG)
from owmtl.icbhi_data import load_split

split_map, split_info = load_split(mode="patient_independent")
print("split file :", split_info["path"])
print("policy     :", split_info["split_method"])
print("recordings :", split_info["n_train"], "train /", split_info["n_test"], "test",
      f"({split_info['test_fraction'] * 100:.1f}% test)")
print("patients   :", split_info["n_patients"],
      "| moved to train to remove leakage:", split_info["leaking_patients"])

# Fail loudly rather than run on a wrong split -- that is the failure this cell replaces.
assert split_info["n_recordings"] == 920, split_info["n_recordings"]
assert split_info["is_patient_independent"], split_info["leaking_patients_still_present"]

SPLIT_FILE = "/kaggle/working/ICBHI_challenge_train_test.txt"
with open(SPLIT_FILE, "w") as fh:
    for stem in sorted(split_map):
        fh.write(f"{stem}\t{split_map[stem]}\n")
print("wrote", SPLIT_FILE)

split file : /kaggle/input/datasets/barshonbasak/owmtl-package/owmtl/ICBHI_challenge_train_test.txt
policy     : patient_independent_official_60_40
recordings : 551 train / 369 test (40.1% test)
patients   : 126 | moved to train to remove leakage: ['156', '218']
wrote /kaggle/working/ICBHI_challenge_train_test.txt


In [2]:
# --- setup: make the owmtl package importable ---------------------------------
# On Kaggle: upload the `owmtl/` folder as a dataset and point OWMTL_PKG at it,
# or place owmtl/ next to this notebook.
import sys, os
OWMTL_PKG = "/kaggle/input/datasets/barshonbasak/owmtl-package"          # TODO: path that CONTAINS the `owmtl` folder
sys.path.insert(0, OWMTL_PKG)

import numpy as np, json, time
from owmtl import icbhi_data as D
from owmtl.concept_extractors import (extract_concept_vector, CONCEPT_NAMES,
                                       CONCEPT_LABEL_MAP, ConceptConfig)
from owmtl import eval_utils as E
print("concepts:", CONCEPT_NAMES)


concepts: ['crackle_presence', 'crackle_rate_hz', 'fine_crackle_ratio', 'coarse_crackle_ratio', 'wheeze_presence', 'wheeze_dominant_freq_hz', 'wheeze_duration_ratio', 'rhonchi_presence', 'spectral_flatness', 'papr_db', 'inspiratory_energy_fraction', 'transient_timing_centroid', 'dominant_freq_hz', 'low_high_freq_ratio']


In [3]:
# --- CONFIG: point these at the real ICBHI files -----------------------------
AUDIO_DIR = "/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files"

# Written by cell 1 from the packaged official split (NOT hand-typed).
SPLIT_FILE = "/kaggle/working/ICBHI_challenge_train_test.txt"

DIAG_FILE  = "/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv"

SR = 16000
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)


### CC2 — device-structure feasibility (Gate G4). Cheap; decides the device axis now.

In [4]:
from owmtl.device_check import analyze, format_report
dev_report = analyze(AUDIO_DIR, DIAG_FILE)
print(format_report(dev_report))
with open(os.path.join(OUT_DIR, "device_structure_report.json"), "w") as fh:
    json.dump(dev_report, fh, indent=2)


=== ICBHI device-structure feasibility (Gate G4) ===
files parsed        : 920/920
patients            : 126
devices (4): {'AKGC417L': 646, 'Meditron': 127, 'LittC2SE': 87, 'Litt3200': 60}
multi-device patients: 4
device-confounded dx : {'Bronchiectasis': {'device': 'Meditron', 'fraction': 1.0}, 'URTI': {'device': 'Meditron', 'fraction': 1.0}, 'Bronchiolitis': {'device': 'Meditron', 'fraction': 1.0}, 'Healthy': {'device': 'Meditron', 'fraction': 1.0}, 'Asthma': {'device': 'LittC2SE', 'fraction': 1.0}, 'LRTI': {'device': 'Meditron', 'fraction': 1.0}}

VERDICT: LODO UNDERPOWERED: only 4 patient(s) span >1 device, so a leave-one-device-out contrast is almost entirely a leave-those-patients-out contrast. Do not make a device claim from it. Also: 6 diagnosis(es) are device-confounded (Bronchiectasis, URTI, Bronchiolitis, Healthy, Asthma, LRTI).


### Build the cycle index on the OFFICIAL split (patient-independent by construction).

In [5]:
records = D.build_cycle_index(AUDIO_DIR, SPLIT_FILE, DIAG_FILE)
from collections import Counter
print("cycles:", len(records))
print("split :", Counter(r.split for r in records))
print("sound :", Counter(r.sound_label for r in records), "(0=N,1=C,2=W,3=B)")
print("device:", Counter(r.device for r in records))


cycles: 6898
split : Counter({'train': 4262, 'test': 2636})
sound : Counter({0: 3642, 1: 1864, 2: 886, 3: 506}) (0=N,1=C,2=W,3=B)
device: Counter({'AKGC417L': 4346, 'Meditron': 1456, 'LittC2SE': 594, 'Litt3200': 502})


### Extract the physics concept vector for every cycle (label-free DSP).

In [6]:
cfg = ConceptConfig(sr=SR)
X = np.zeros((len(records), len(CONCEPT_NAMES)), dtype=np.float32)
meta = {k: [] for k in ("patient","device","split","crackle","wheeze","sound_label","diagnosis")}
t0 = time.time()
for i, r in enumerate(records):
    try:
        y = D.load_cycle_waveform(AUDIO_DIR, r, sr=SR)
        X[i] = extract_concept_vector(y, SR, cfg)
    except Exception as ex:
        X[i] = 0.0
        if i < 5: print("warn", r.stem, ex)
    for k in meta: meta[k].append(getattr(r, k))
    if (i+1) % 500 == 0: print(f"{i+1}/{len(records)}  ({time.time()-t0:.0f}s)")
meta = {k: np.array(v) for k, v in meta.items()}
np.savez_compressed(os.path.join(OUT_DIR, "concepts_all.npz"),
                    X=X, concept_names=np.array(CONCEPT_NAMES), **meta)
print("saved concepts_all.npz  shape", X.shape)


500/6898  (19s)
1000/6898  (27s)
1500/6898  (36s)
2000/6898  (45s)
2500/6898  (53s)
3000/6898  (62s)
3500/6898  (71s)
4000/6898  (81s)
4500/6898  (91s)
5000/6898  (100s)
5500/6898  (107s)
6000/6898  (117s)
6500/6898  (125s)
saved concepts_all.npz  shape (6898, 14)


### Gate G2 — validate concepts against ICBHI labels
`crackle_presence` vs the ICBHI `crackle` bit, `wheeze_presence` vs `wheeze` bit, on the **test** split, with
bootstrap 95% CIs (Protocol Essential #10). AUROC clearly > 0.5 ⇒ the physics concept tracks the clinical label.

In [ ]:
test = meta["split"] == "test"
report = {"gate": "G2", "n_test_cycles": int(test.sum()), "concept_label_auroc": {}}
name_idx = {n: i for i, n in enumerate(CONCEPT_NAMES)}
for concept, labelname in CONCEPT_LABEL_MAP.items():
    y = meta[labelname][test].astype(int)
    s = X[test, name_idx[concept]]
    pt, lo, hi = E.auroc_ci(y, s, n_boot=1000)
    report["concept_label_auroc"][concept] = {"vs_label": labelname,
        "auroc": round(pt,4), "ci95": [round(lo,4), round(hi,4)],
        "n_pos": int(y.sum()), "n_neg": int((1-y).sum())}
    print(f"{concept:18s} vs {labelname:8s}: AUROC {pt:.3f}  95% CI [{lo:.3f},{hi:.3f}]  (n+={y.sum()})")

# also: how well the full concept vector linearly separates each sound label (sanity)
# StandardScaler is not cosmetic here: low_high_freq_ratio spans 0.018-18910 while
# crackle_presence is in [0,1], so unscaled lbfgs hits max_iter without converging and the
# AUROC it reports is wherever the optimiser happened to stop (0.61 vs 0.66 across two runs
# of the same data). Scale, and give it room to converge.
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
tr = ~test
for name, lab in [("crackle","crackle"),("wheeze","wheeze")]:
    try:
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=2000, class_weight="balanced")
                            ).fit(X[tr], meta[lab][tr])
        assert clf[-1].n_iter_[0] < 2000, f"{lab}: still not converged"
        auc = roc_auc_score(meta[lab][test], clf.predict_proba(X[test])[:,1])
        report.setdefault("full_vector_auroc", {})[lab] = round(float(auc),4)
        print(f"full concept vector -> {lab}: AUROC {auc:.3f}")
    except Exception as ex:
        print("skip", lab, ex)

with open(os.path.join(OUT_DIR, "concept_validation_report.json"), "w") as fh:
    json.dump(report, fh, indent=2)


### G2 decision
- **PASS** (single-concept AUROC materially > 0.5, e.g. ≳ 0.6, CI lower bound > 0.5): the physics concepts are
  clinically meaningful → proceed to notebook 02 (bottleneck). The full-vector AUROCs should be higher still.
- **BORDERLINE/FAIL:** shrink to the most reliable concepts (PAPR, spectral_flatness, wheeze band) or add a
  learned concept-refinement step; re-validate before building the bottleneck.

This report (`concept_validation_report.json`) is the evidence a reviewer will ask for that "physics concepts" are real.
